# R19-H206 - The catalogue-code arm

Catalogue codes are 61% of the wide probe set and the H194 router's blind spot: the value comparator
abstains (no unit), word-overlap's `[a-z][a-z0-9-]{2,}` token regex silently drops leading-digit
identifiers. This notebook (a) builds a 30-pair blind-adjudicated code stratum and an exact-identifier
router arm (alphanumeric-preserving, case/hyphen/space-folded, code-shaped token anchoring), and (b)
tests the query-side twin: rewriting device-scoped code queries to seed on the DEVICE entity to recover
the present-but-unranked code golds identified by the H207 canonical census.

neo4j2 READ-ONLY; CPU-only; retrieval pinned to an explicit neo4j2 driver. Uses the H207 canonical
render spec.


In [1]:
# CPU-only; pin retrieval to neo4j2 via an explicit driver (NOT .env / Foundry defaults)
import os
os.environ["CUDA_VISIBLE_DEVICES"] = ""                     # CPU-only
import re, json, pickle, hashlib, itertools, unicodedata, datetime
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np
from neo4j import GraphDatabase
from knowledge_graph_foundry import load_settings
from knowledge_graph_foundry.graph.graphrag import vector_query
from rich import print as rprint

ROOT = Path("..")
NEO4J2 = "bolt://user-konrad.jelen-kgf-neo4j2:7687"         # pinned baseline, READ-ONLY
DEFAULT_NEO4J = "bolt://user-konrad.jelen-kgf-neo4j:7687"   # the .env default (the trap)
AUTH = ("neo4j", os.environ.get("NEO4J_PASSWORD", "kgfoundry"))
settings = load_settings(ROOT / "config.yml")
VEC = settings.graphrag.vector_index_name
K64, RETRIEVE_TOPK, REL_LIMIT = 64, 128, 15
driver2 = GraphDatabase.driver(NEO4J2, auth=AUTH)           # every vector_query uses THIS driver
rprint(f"[cyan]config[/cyan] vec={VEC} eval_k={K64} retrieve_top_k={RETRIEVE_TOPK} rel_cap={REL_LIMIT} (CPU-only, neo4j2 pinned)")


2026-07-07 18:01:22.561 | INFO     | knowledge_graph_foundry.config:<module>:40 - PROJ_ROOT path is: /home/lab/workspace/learning/projects/knowledge-graph-foundry


config vec=kgf_entity_embeddings eval_k=64 retrieve_top_k=128 rel_cap=15 (CPU-only, neo4j2 pinned)

In [2]:
# Graph pull from neo4j2 (READ-ONLY) + production-faithful render primitives
# Mirrors pipeline._retrieve_local entity_blocks: name/aka/description/Properties/Relations.
with driver2.session() as s:
    ents = s.run("MATCH (e:Entity) RETURN e.id AS id, e.name AS name, e.description AS description, "
                 "properties(e) AS props, labels(e) AS types").data()
    edges = s.run("MATCH (a:Entity)-[r]-(b:Entity) WHERE type(r)<>'SIMILAR_TO' AND a.id<b.id "
                  "RETURN DISTINCT a.id AS a, b.id AS b, type(r) AS rel").data()
    prop_rows = s.run("MATCH (p:Proposition)-[:ABOUT]->(e:Entity) RETURN e.id AS eid, p.text AS text").data()
    alias_rows = s.run("MATCH (e:Entity)-[:SAME_AS*1..2]-(a:Entity) WHERE e.id<>a.id "
                       "RETURN e.id AS eid, collect(DISTINCT a.id)[..5] AS aliases").data()
    emb_head = {r["id"]: r["head"] for r in s.run(
        "MATCH (e:Entity) WHERE e.embedding IS NOT NULL RETURN e.id AS id, e.embedding[0..8] AS head").data()}
node = {r["id"]: r for r in ents}; names = {r["id"]: r["name"] for r in ents}
props_by = defaultdict(list); [props_by[r["eid"]].append(r["text"]) for r in prop_rows]
alias_by = {r["eid"]: r["aliases"] for r in alias_rows}
rels_by = defaultdict(list)
for e in edges:
    rels_by[e["a"]].append((e["rel"], e["b"])); rels_by[e["b"]].append((e["rel"], e["a"]))

def spec_of(r): return {k.removeprefix("prop_"): v for k, v in r["props"].items() if k.startswith("prop_")}
def merged_spec(nid):
    r = node[nid]; spec = dict(spec_of(r))
    for a in [a for a in alias_by.get(nid, []) if a in node]:
        for k, v in spec_of(node[a]).items(): spec.setdefault(k, v)
    return spec
def base_render(nid):
    r = node[nid]; spec = merged_spec(nid); al = [a for a in alias_by.get(nid, []) if a in node]
    aka = (f"Also known as: {', '.join(names.get(a, '') for a in al)}\n" if al else "")
    return f"## {r['name']} ({', '.join(r['types'])})\n{aka}{r['description'] or ''}\nProperties: {json.dumps(spec, default=str)}"
def seed_render(nid):
    rels = "; ".join(f"{t} -> {names.get(b, '')}" for t, b in rels_by.get(nid, [])[:REL_LIMIT])
    return base_render(nid) + "\nRelations: " + rels
def units_of(ids): return [seed_render(n) for n in ids] + [t for n in ids for t in props_by.get(n, [])]
rprint(f"[green]pulled neo4j2[/green] entities {len(node)} edges {len(edges)} propositions {len(prop_rows)}")


pulled neo4j2 entities 2798 edges 3905 propositions 19654

In [3]:
# ---- scorers: H194 router (verbatim) + deterministic presence (exact U word-overlap) ----
_TM = dict.fromkeys(map(ord, "®™©"), None)
def gnorm(s):
    s = (s or "").translate(_TM); s = unicodedata.normalize("NFKC", s)
    s = s.replace(" ", " ").replace("×", "x").replace("*", "x").replace("·", "x")
    s = re.sub(r"(?<=\d),(?=\d)", "", s)
    return re.sub(r"\s+", " ", s.casefold()).strip()
UNITWORD = {"mm":"len_mm","cm":"len_cm","g":"mass_g","kg":"mass_kg","oz":"mass_oz","ml":"vol_ml","l":"vol_l",
   "db":"sound_db","dba":"sound_db","w":"power_w","hz":"freq_hz","cmh2o":"press","m":"alt_m",
   "min":"time_min","mins":"time_min","minute":"time_min","minutes":"time_min","year":"warr_y","years":"warr_y"}
FAM_EQ = {"len_mm":{"len_mm"},"len_cm":{"len_cm"},"alt_m":{"alt_m"},"mass_g":{"mass_g"},"mass_kg":{"mass_kg"},
   "mass_oz":{"mass_oz"},"vol_ml":{"vol_ml","vol_l"},"sound_db":{"sound_db"},"power_w":{"power_w"},
   "time_min":{"time_min"},"warr_y":{"warr_y"},"press":{"press"},"freq_hz":{"freq_hz"}}
def key_family(k):
    k = k.lower()
    if "dimension" in k or re.search(r"_mm\b", k) or "length_mm" in k: return "len_mm"
    if "altitude" in k: return "alt_m"
    if k.endswith("_kg") or "weight_kg" in k: return "mass_kg"
    if re.search(r"_g\b", k): return "mass_g"
    if "_oz" in k: return "mass_oz"
    if re.search(r"_ml\b", k) or "capacity_ml" in k or "water" in k: return "vol_ml"
    if "sound" in k or re.search(r"_db\b", k) or "noise" in k: return "sound_db"
    if "power" in k or "consumption" in k: return "power_w"
    if "ramp" in k or "delay" in k: return "time_min"
    if "warranty" in k: return "warr_y"
    if "pressure" in k: return "press"
    return None
def nums_in(v): return re.findall(r"\d+(?:\.\d+)?", str(v).replace(",", ""))
def ctx_quantities(ids):
    Q = set()
    for nid in ids:
        spec = merged_spec(nid); unit_for = {}
        for k, v in spec.items():
            if k.endswith("_unit"):
                fam = UNITWORD.get(gnorm(str(v)).replace(" ", ""))
                if fam: unit_for[k[:-5]] = fam
        for k, v in spec.items():
            fam = key_family(k) or unit_for.get(k)
            if fam:
                for n in nums_in(v): Q.add((n, fam))
        text = gnorm(seed_render(nid) + " " + " ".join(props_by.get(nid, [])))
        for m in re.finditer(r"(\d+(?:\.\d+)?)\s*(mm|cm|dba|db\(a\)|db|kg|oz|ml|cmh2o|hz|mins|minutes|minute|min|years|year|w|g|l|m)\b", text):
            fam = UNITWORD.get(m.group(2).replace("(a)", ""))
            if fam: Q.add((m.group(1), fam))
    return Q
def parse_gold(gold):
    g = gnorm(gold)
    if re.search(r"\d+\s*x\s*\d+\s*x\s*\d+", g): return ("dim", re.findall(r"\d+(?:\.\d+)?", g))
    if "sd card" in g: return ("sdcard", None)
    m = re.search(r"(\d+(?:\.\d+)?)\s*(mm|cm|dba|db\(a\)|db|kg|oz|ml|cmh2o|cm h2o|hz|mins|minutes|minute|min|years|year|w|g|l|m)\b", g)
    rng = re.search(r"(\d+(?:\.\d+)?)\s*(?:to|-)\s*(\d+(?:\.\d+)?)", g)
    if m:
        u = m.group(2).replace("(a)", "").replace("cm h2o", "cmh2o").replace(" ", ""); fam = UNITWORD.get(u)
        if rng and rng.group(2): return ("range", (rng.group(1), rng.group(2), fam))
        return ("num", (m.group(1), fam))
    if rng and rng.group(2):
        fam = "press" if "cmh2o" in g or "cm h2o" in g else ("time_min" if "min" in g else None)
        return ("range", (rng.group(1), rng.group(2), fam))
    return ("other", None)
def comparator(gold, ids):
    kind, payload = parse_gold(gold); Q = ctx_quantities(ids)
    T = gnorm(" ".join(seed_render(n) + " " + " ".join(props_by.get(n, [])) for n in ids))
    if kind == "dim":
        a, b, c = payload; mm = {n for n, f in Q if f == "len_mm"}
        if {a, b, c} <= mm: return True
        t = T.replace(" ", "")
        return any(re.search(r"(?<!\d)" + p[0] + "x" + p[1] + "x" + p[2] + r"(?!\d)", t) for p in itertools.permutations([a, b, c]))
    if kind == "num":
        n, fam = payload
        if fam is None: return any(x == n for x, _ in Q)
        eq = FAM_EQ.get(fam, {fam}); return any(x == n and f in eq for x, f in Q)
    if kind == "range":
        a, b, fam = payload; t = T.replace(" ", "")
        if re.search(r"(?<!\d)" + a + r"\s*-\s*" + b, T) or (a + "-" + b) in t or (a + "to" + b) in t: return True
        if fam:
            eq = FAM_EQ.get(fam, {fam}); xs = {x for x, f in Q if f in eq}; return a in xs and b in xs
        return False
    if kind == "sdcard":
        t = T.replace(" ", ""); return ("sdcard" in t) and (">1year" in t or "1year" in t)
    return None
def word_overlap(gold, ids, thr=0.6):
    ng = gnorm(gold); ctx = gnorm(" ".join(units_of(ids)))
    w = set(re.findall(r"[a-z][a-z0-9\-]{2,}", ng)); cw = set(re.findall(r"[a-z][a-z0-9\-]{2,}", ctx))
    return bool(w) and len(w & cw) / len(w) >= thr
def router_present(gold, ids):
    r = comparator(gold, ids)
    if r is None: return bool(word_overlap(gold, ids))
    return bool(r)

# deterministic exact-value presence (H193/H194 robustness instrument; CPU-only, glyph/unit aware)
GLYPH = {'™':'', '®':'', '©':'', '–':'-', '—':'-', ' ':' ', ' ':' ',
         ' ':' ', 'ﬁ':'fi', 'ﬂ':'fl', '′':"'", '″':'"', '°':' ', '×':'x'}
def gnorm2(s):
    s = s or ""
    for k, v in GLYPH.items(): s = s.replace(k, v)
    s = unicodedata.normalize("NFKD", s); s = "".join(c for c in s if not unicodedata.combining(c))
    return re.sub(r"\s+", " ", s).casefold().strip()
_UNIT = r"(cmh2o|cm h2o|mm|cm|dba|db\(a\)|db|kg|g|oz|ml|l|w|hz|watts?|mins?|hours?|years?|m)"
def _numunits(t): return re.findall(r"(\d[\d,\.]*)\s*" + _UNIT + r"?", gnorm2(t))
def exact_present(gold, ctx):
    g = gnorm2(gold); c = gnorm2(ctx)
    if g and g in c: return True
    gd = re.sub(r"[ ,]", "", g); cd = re.sub(r"[ ,]", "", c)
    if any(ch.isdigit() for ch in gd) and len(gd) >= 4 and gd in cd: return True
    gnu = _numunits(gold)
    if gnu:
        for num, unit in gnu:
            nd = num.replace(",", "")
            if unit:
                if not (re.search(r"(?<!\d)" + re.escape(nd) + r"\s*" + re.escape(unit), c) or
                        re.search(re.escape(nd) + re.escape(unit), cd)): return False
            else:
                if not re.search(r"(?<!\d)" + re.escape(nd) + r"(?!\d)", cd): return False
        return True
    return False
def is_numeric_gold(g): return bool(re.search(r"\d", g))
def deterministic_present(gold, ids):
    ctx = " ".join(units_of(ids))
    if is_numeric_gold(gold): return exact_present(gold, ctx)
    if exact_present(gold, ctx): return True
    return word_overlap(gold, ids)
rprint("[green]scorers ready[/green] router (comparator+word-overlap)  |  deterministic (exact U word-overlap, CPU-only)")


scorers ready router (comparator+word-overlap)  |  deterministic (exact U word-overlap, CPU-only)

In [4]:
# ---- BLIND CODE-STRATUM BENCH (built + adjudicated BEFORE the arm is defined) ----
# H190 glyph rules for identifier normalization (case/hyphen/space-folded, alphanumeric-preserving).
GLYPH = {'™':'', '®':'', '©':'', '–':'-', '—':'-', ' ':' ', ' ':' ',
         ' ':' ', 'ﬁ':'fi', 'ﬂ':'fl', '′':"'", '″':'"', '°':' ', '×':'x'}
def codenorm(s):
    s = s or ""
    for k, v in GLYPH.items(): s = s.replace(k, v)
    s = unicodedata.normalize("NFKD", s); s = "".join(c for c in s if not unicodedata.combining(c))
    return re.sub(r"[\s\-]", "", s).casefold()          # fold case/hyphen/space, keep alphanumerics
def code_tokens(text):
    return {codenorm(t) for t in re.findall(r"[A-Za-z0-9][A-Za-z0-9\-]{2,}", text or "")}

# graph-wide code-token index = independent adjudication oracle (arm-agnostic).
# Keep an ORIGINAL spelling per normalized token so harvested codes render faithfully.
CODE_SHAPE = re.compile(r"^[A-Za-z]{0,2}\d{3,}[A-Za-z0-9]*$")   # catalogue/HCPCS/SKU shape: E0471, PS0001464, 1097348, WM31660
GRAPH_CODE_TOKENS = set()
carrier_of = {}                                          # codenorm -> an entity that carries it
orig_of = {}                                             # codenorm -> original spelling
for nid in node:
    txt = seed_render(nid) + " " + " ".join(props_by.get(nid, []))
    for t in re.findall(r"[A-Za-z0-9][A-Za-z0-9\-]{2,}", txt):
        tk = codenorm(t)
        GRAPH_CODE_TOKENS.add(tk)
        carrier_of.setdefault(tk, nid); orig_of.setdefault(tk, t)

# present set: wide-probe catalogue codes first, then supplement with graph-harvested catalogue-shaped
# codes, to a 30-code blind stratum. All adjudicated present by the arm-agnostic oracle.
cat_codes = sorted({p["gold_evidence"][0] for p in json.load(open(ROOT / "data/processed/probes-wide-h188.json"))["probes"]
                    if p["derivation_rule"] == "catalogue_code"})
present_codes = [c for c in cat_codes if codenorm(c) in GRAPH_CODE_TOKENS]
seen = {codenorm(c) for c in present_codes}
for tk in sorted(GRAPH_CODE_TOKENS):
    if len(present_codes) >= 30: break
    o = orig_of[tk]
    if tk not in seen and CODE_SHAPE.match(o.replace("-", "")):
        present_codes.append(o); seen.add(tk)
present_codes = present_codes[:30]
def make_lookalike(code):
    for i in range(len(code) - 1, -1, -1):
        if code[i].isdigit():
            for d in "5060342178":
                cand = code[:i] + d + code[i + 1:]
                if codenorm(cand) != codenorm(code) and codenorm(cand) not in GRAPH_CODE_TOKENS:
                    return cand
    return code + "X9"                                   # fallback: append absent suffix
bench = []
for c in present_codes:
    cid = carrier_of[codenorm(c)]
    ctx = seed_render(cid) + " " + " ".join(props_by.get(cid, []))
    la = make_lookalike(c)
    bench.append(dict(code=c, family=re.sub(r"\d", "#", c), label="present", carrier=cid, context=ctx))
    bench.append(dict(code=la, family=re.sub(r"\d", "#", c), label="absent", carrier=cid, context=ctx,
                      lookalike_of=c))
# freeze blind labels to disk BEFORE the arm runs
outb = ROOT / "data/processed/instrument-code-bench-h206.json"
outb.write_text(json.dumps(dict(protocol="H194 blind-first; oracle=graph code-token index (arm-agnostic)",
    n_pairs=len(present_codes), n_cases=len(bench),
    cases=[{k: b[k] for k in b if k != "context"} for b in bench]), indent=2))
rprint(f"[green]blind code bench frozen[/green] {len(present_codes)} pairs / {len(bench)} cases -> {outb.name}")
rprint(f"  present exemplars: {present_codes[:6]}")
rprint(f"  look-alike exemplars: {[b['code'] for b in bench if b['label']=='absent'][:6]} (same family, absent from graph)")


blind code bench frozen 30 pairs / 60 cases -> instrument-code-bench-h206.json

present exemplars: ['1024582', '1063785', '1067148', '1069193', '1082653', '1097345']

look-alike exemplars: ['1024585', '1063780', '1067145', '1069195', '1082655', '1097340'] (same family, absent 
from graph)

In [5]:
# ---- clause (a): exact-identifier router arm + score the frozen bench ----
def exact_identifier_arm(code, context):
    """Anchored code-shaped token match: normalize (H190 glyph, case/hyphen/space fold, alphanumeric-
    preserving) and require the code to appear as a WHOLE code-shaped token, not a substring.
    Anchoring stops E0471 crediting inside E04715 and stops leading-digit codes being dropped."""
    cn = codenorm(code)
    if not cn or not any(ch.isdigit() for ch in cn):     # only fires on code-shaped (digit-bearing) tokens
        return None
    return cn in code_tokens(context)

agree = 0; false_credit = 0; decisions = 0; per = []
for b in bench:
    pred = exact_identifier_arm(b["code"], b["context"])
    if pred is None:                                     # not code-shaped -> arm abstains (excluded)
        continue
    decisions += 1
    truth = (b["label"] == "present")
    if pred == truth: agree += 1
    if pred and not truth: false_credit += 1             # credited an absent look-alike
    per.append(dict(code=b["code"], label=b["label"], pred=bool(pred)))
agreement = agree / decisions if decisions else 0.0
clause_a = agreement >= 0.95 and false_credit == 0
rprint(f"[bold cyan]clause (a) exact-identifier arm[/bold cyan] decisions={decisions} agreement={agreement:.3f} "
       f"(bar>=0.95) false_credit_on_lookalikes={false_credit} (bar=0)")
rprint(f"  -> [{'green' if clause_a else 'red'}]{'PASS' if clause_a else 'FAIL'}[/]")


clause (a) exact-identifier arm decisions=60 agreement=1.000 (bar>=0.95) false_credit_on_lookalikes=0 (bar=0)

-> PASS

In [6]:
# ---- clause (b): device-seeded query rewrite on the present-but-unranked code golds ----
# Source of truth for "unranked code golds": the H207 canonical census report.
import glob
h207 = json.load(open(sorted(glob.glob(str(ROOT / "reports/render-parity-h207-*.json")))[-1]))
unranked = [m for m in h207["miss_detail"] if m["cls"] == "present-but-unranked" and m["rule"] == "catalogue_code"]
rprint(f"[cyan]present-but-unranked catalogue golds (from H207)[/cyan] n={len(unranked)}: "
       f"{[(m['pid'], m['gold']) for m in unranked]}")

def locate_device(product):
    """Approximate 'vector-search the device name' by locating the entity whose name best matches the
    product string (token overlap), then seed retrieval on THAT entity's own stored embedding."""
    pt = set(re.findall(r"[a-z0-9]{3,}", product.casefold()))
    best, bs = None, 0.0
    for nid, nm in names.items():
        nt = set(re.findall(r"[a-z0-9]{3,}", (nm or "").casefold()))
        if not nt: continue
        j = len(pt & nt) / len(pt | nt)
        if j > bs: best, bs = nid, j
    return best, bs

def device_seeded_render(device_id):
    with driver2.session() as s:
        emb = s.run("MATCH (e:Entity {id:$id}) RETURN e.embedding AS v", id=device_id).single()
    ids = []
    if emb and emb["v"] is not None:
        ids = [x["id"] for x in vector_query(driver2, emb["v"], VEC, top_k=K64) if x["id"] in node]  # unchanged k
    neigh = [b for _, b in rels_by.get(device_id, [])]                # 1-hop graph neighbourhood of the device
    seen, order = set(), []
    for n in [device_id] + ids + neigh:
        if n in node and n not in seen: seen.add(n); order.append(n)
    return order[:K64], units_of(order[:K64])

recovered = []
for m in unranked:
    dev, score = locate_device(m["product"])
    ok = False
    if dev is not None:
        _, ctx_units = device_seeded_render(dev)
        ctx = " ".join(ctx_units)
        ok = (exact_identifier_arm(m["gold"], ctx) is True) or exact_present(m["gold"], ctx)
    recovered.append(dict(pid=m["pid"], code=m["gold"], product=m["product"][:40],
                          device=names.get(dev, None), match=round(score, 2), recovered=bool(ok)))
    rprint(f"  {m['pid']} {m['gold']:<10} device='{names.get(dev,'?')[:32]}' (match {score:.2f}) "
           f"-> [{'green' if ok else 'red'}]{'RECOVERED' if ok else 'still unranked'}[/]")
n_rec = sum(1 for r in recovered if r["recovered"])
rate = n_rec / len(unranked) if unranked else 0.0
clause_b = rate >= 0.50
rprint(f"[bold cyan]clause (b) device-seeded rewrite[/bold cyan] recovered {n_rec}/{len(unranked)} = {rate:.1%} "
       f"(bar>=50%) -> [{'green' if clause_b else 'red'}]{'PASS' if clause_b else 'FAIL (render-budget refuter)'}[/]")


present-but-unranked catalogue golds (from H207) n=5: [('W004', 'PS0001464'), ('W010', '1097348'), ('W012', 
'PS0001231'), ('W063', 'P1295'), ('W066', 'WM31660')]

W004 PS0001464  device='Continuous Positive Airway Press' (match 1.00) -> still unranked

W010 1097348    device='Pro-Flow nasal cannula' (match 0.80) -> still unranked

W012 PS0001231  device='Automatic Positive Airway Pressu' (match 0.60) -> still unranked

W063 P1295      device='Nasal cannula' (match 0.50) -> still unranked

W066 WM31660    device='DC Adapter 12/24 V' (match 1.00) -> RECOVERED

clause (b) device-seeded rewrite recovered 1/5 = 20.0% (bar>=50%) -> FAIL (render-budget refuter)

In [7]:
# ---- report + verdict ----
stamp = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
verdict = ("CONFIRMED" if (clause_a and clause_b) else
           "PARTIAL" if clause_a else "REFUTED")
report = dict(
    hypothesis="R19-H206", utc=stamp, graph="neo4j2", read_only=True, cpu_only=True,
    uses_h207_canonical_spec=True,
    clause_a=dict(name="exact-identifier router arm on blind code stratum",
                  n_pairs=len(present_codes), n_decisions=decisions, agreement=float(agreement),
                  false_credit_on_lookalikes=int(false_credit), bar="agreement>=0.95 & false_credit==0",
                  pass_=bool(clause_a), bench_file="data/processed/instrument-code-bench-h206.json",
                  per_case=per),
    clause_b=dict(name="device-seeded query rewrite on unranked code golds",
                  n_unranked=len(unranked), recovered=int(n_rec), recovery_rate=float(rate),
                  bar="recover>=50%", pass_=bool(clause_b), detail=recovered,
                  refuter="if <50%, codes are render-budget victims -> lever moves to render policy"),
    verdict=verdict)
outp = ROOT / f"reports/code-arm-h206-{stamp}.json"
outp.write_text(json.dumps(report, indent=2, default=str))
driver2.close()
rprint(f"[green]report written[/green] {outp}")
rprint(f"[bold green]===== R19-H206 VERDICT: {verdict} =====[/]  clause(a)={'PASS' if clause_a else 'FAIL'}  "
       f"clause(b)={'PASS' if clause_b else 'FAIL'}")


report written ../reports/code-arm-h206-20260707T160127Z.json

===== R19-H206 VERDICT: PARTIAL =====  clause(a)=PASS  clause(b)=FAIL